## Differential Expression

This notebook identifies genes whose expression differs between normal and tumor breast tissue samples in the cleaned dataset produced earlier in the project.

The analysis is intentionally simple and transparent: we compare the two groups gene by gene, estimate the direction and magnitude of the difference, test whether that difference is likely to reflect more than sampling noise, and then correct for the fact that thousands of genes are tested at once.

Main steps:
1. split the samples into tumor and normal groups
2. compute the mean expression of each gene in each group
3. estimate the expression difference on the log scale
4. run a statistical test for each gene
5. correct p-values with the Benjamini-Hochberg FDR procedure
6. save complete and filtered result tables for downstream interpretation and visualization

In [1]:
# imports

import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

In [2]:
# Paths
project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

# Get data path
data_path = str(project_root) + "/data"

# Get results path
result_path = str(project_root) + "/results"

# Step 1: Split tumor vs normal

Differential expression is defined relative to a comparison, so the first step is to separate the samples into the two biological conditions of interest: `tumor` and `normal`.

The cleaned dataset already excludes `cell_line` samples and uses a binary label created in Notebook 01. Restricting the analysis to primary tissue samples keeps the comparison biologically coherent, because in vitro cell lines can have transcriptional profiles that differ substantially from patient tissue.

After filtering, the dataset contains 130 tumor samples and 7 normal samples. This imbalance is important and should be kept in mind when interpreting the results, especially for variance estimation and statistical power.

In [3]:
X = pd.read_csv(data_path + "/processed/X_clean.csv")
y = pd.read_csv(data_path + "/processed/y_clean.csv").squeeze()

In [4]:
X_tumor = X[y == 'tumor']
X_normal = X[y == 'normal']

# Step 2: Compute mean expression

For each gene, we compute the mean expression across tumor samples and across normal samples.

These two group means provide a compact summary of the typical expression level in each condition. They are useful both for interpretation and for the next step, where we quantify how large the between-group difference is.

In [5]:
mean_tumor = X_tumor.mean(axis=0)
mean_normal = X_normal.mean(axis=0)

# Step 3: Compute log2 fold change

The expression values were previously assessed as already normalized and likely log-transformed. On a log2 scale, subtracting the group means gives a log-scale expression difference.

This value can be interpreted as a log2 fold change summary: positive values indicate higher expression in tumors, while negative values indicate higher expression in normal tissue. For example, a log2 fold change of `+1` corresponds to roughly a two-fold higher expression in tumors, whereas `-1` corresponds to roughly a two-fold higher expression in normal samples.

In [6]:
log2_fc = mean_tumor - mean_normal

# Step 4: Statistical test

A difference in mean expression is not enough by itself: some genes can show an apparent shift simply because of biological variability or sampling noise. We therefore test each gene statistically.

For every gene, the null hypothesis is that the mean expression is the same in the tumor and normal groups. The test returns a p-value measuring how compatible the observed difference is with that null hypothesis.

We use Welch's t-test (`equal_var=False`) rather than the classical Student t-test. This is the appropriate choice here because the groups are strongly imbalanced (130 tumors versus 7 normals) and their variances do not need to be assumed equal. Welch's test is more robust under unequal variances and unequal sample sizes.

This remains a simple univariate screening approach. With only 7 normal samples, the estimated variance in that group is less stable, so p-values should be interpreted with appropriate caution.

In [7]:
p_values = []

for gene in X.columns:
    stat, p = ttest_ind(
        X_tumor[gene],
        X_normal[gene],
        equal_var=False  # important (Welch t-test)
    )
    p_values.append(p)

p_values = pd.Series(p_values, index=X.columns)

# Step 5: Multiple testing correction

We test 54,675 genes, so some small p-values are expected to appear by chance even if no real biological difference exists. Using raw p-values alone would therefore produce many false positives.

To address this, we apply the Benjamini-Hochberg false discovery rate (FDR) correction. The adjusted p-value estimates significance while accounting for the large number of simultaneous tests.

FDR control is usually preferred in transcriptomic studies because it balances discovery and reliability. It is less conservative than family-wise error rate procedures such as Bonferroni, which can become overly strict when tens of thousands of genes are tested and may hide genuinely relevant signals.

In [8]:
adj_p_values = multipletests(p_values, method='fdr_bh')[1]
adj_p_values = pd.Series(adj_p_values, index=X.columns)

# Step 6: Save results

The final step is to assemble the main statistics into one clean results table.

For each gene, we save the mean expression in tumors, the mean expression in normals, the log2-scale difference, the raw p-value, and the FDR-adjusted p-value. Keeping all metrics together makes downstream work easier: the table can be sorted, filtered, plotted in volcano plots, or used to inspect candidate genes manually.

In [9]:
results = pd.DataFrame({
    'mean_tumor': mean_tumor,
    'mean_normal': mean_normal,
    'log2FC': log2_fc,
    'p_value': p_values,
    'adj_p_value': adj_p_values
})
results.to_csv(result_path + "/tables/differential_expression_results.csv")


## Step 7: Save significant, upregulated, and downregulated gene lists

In addition to the full results table, we save filtered gene lists that are easier to interpret biologically.

A gene is kept in the `significant` table only if it passes two thresholds: an adjusted p-value below `0.05` and an absolute log2 fold change above `1`.

Using both criteria helps retain genes that are statistically credible and large enough in effect size to be biologically meaningful. A gene can be statistically significant with a very small effect, especially when many samples are available, so combining significance and effect size produces a more useful shortlist. Here, `|log2FC| > 1` corresponds to at least an approximate two-fold difference between groups.

Finally, we split the significant genes into `upregulated` and `downregulated` sets. Because `log2FC` was defined as `mean_tumor - mean_normal`, positive values correspond to genes more highly expressed in tumors, while negative values correspond to genes more highly expressed in normal tissue.

This separation is useful for downstream interpretation. Genes upregulated in tumors can point to cancer-associated pathways or markers of proliferation, whereas genes downregulated in tumors can highlight normal tissue functions that are reduced or lost in the disease state.

In [14]:
significant = results[
    (results['adj_p_value'] < 0.05) &
    (abs(results['log2FC']) > 1)
]

upregulated = significant[significant['log2FC'] > 0]
downregulated = significant[significant['log2FC'] < 0]

In [15]:
significant.to_csv(result_path + "/tables/significant_genes.csv")
upregulated.to_csv(result_path + "/tables/upregulated_genes.csv")
downregulated.to_csv(result_path + "/tables/downregulated_genes.csv")

In [16]:
print("Results :", len(results))
print("Significant :", len(significant))
print("Upregulated :", len(upregulated))
print("Downregulated :", len(downregulated))

Results : 54675
Significant : 5663
Upregulated : 3750
Downregulated : 1913


## Interpretation of the thresholded results

Applying the thresholds `adj_p_value < 0.05` and `|log2FC| > 1` yields 5,663 differentially expressed genes out of 54,675 tested. Among them, 3,750 are upregulated in tumors and 1,913 are downregulated.

This large number of significant genes indicates a strong transcriptomic difference between normal and tumor tissue. That result is consistent with the PCA analysis from the previous notebook, where the two groups already showed a clear global separation. In other words, the differential expression analysis confirms at the gene level what PCA suggested at the whole-transcriptome level.

The fact that more genes are upregulated in tumors than downregulated is biologically plausible. Tumors often activate many processes simultaneously, including proliferation, metabolism, stress response, and signaling programs. Downregulation is also present, but in this dataset the dominant pattern appears to be broad transcriptional activation in the tumor group.

The chosen thresholds aim to balance statistical confidence and biological interpretability. The FDR cutoff controls the expected proportion of false discoveries among the reported genes, while the `|log2FC| > 1` cutoff focuses attention on genes with at least an approximate two-fold difference, which is easier to interpret than very small but statistically significant shifts.

At the same time, the results should be interpreted with caution because the normal group contains only 7 samples. This small reference group can make variance estimates less stable and can affect which genes pass the significance threshold. The signal is clearly real, but the exact ranking and boundary of significant genes should not be treated as perfectly definitive.
